In [68]:
# Now try to calculate the l=2 mode of the remote quadrupole field
from sympy import *
import numpy as np
import my_remote_spectra as rs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [69]:
import param
import config as config
import kszpsz_config as kszpsz_config

Omega_b = config.Omega_b
Omega_c = config.Omega_c
w = config.w
wa = config.wa
Omega_K = config.Omega_K
h = config.h
As= config.As
ns = config.ns

A = 1
B = 0
r_H = param.r_H #Mpc
z_c = 2
Z_e = 1
a_e = 1/(1+Z_e)
a_dec = kszpsz_config.adec

x_c = rs.chifromz(z_c) #Mpc
r_e = rs.chifromz(Z_e)
r_edec = rs.chifromz(config.zdec) - rs.chifromz(Z_e)
r_dec = rs.chifromz(config.zdec)
D_psi_dec = rs.Dpsi_inter(Omega_b, Omega_c, w, wa, Omega_K, h)(a_dec)
D_v_dec = rs.Dv_inter(Omega_b, Omega_c, w, wa, Omega_K, h)(a_dec)

In [70]:
# test calling symbolic function
def Cos_theta_c():
    str_Cos_theta_c = '(chi_c - chi_e*cos(theta_e))/(Delta_chi_dec)'
    return sympify(str_Cos_theta_c, evaluate=False)

def Delta_cos_theta_c_n(n):
    str_Delta_cos_theta_c_n = f'1 - ({Cos_theta_c()})**{n}'
    return sympify(str_Delta_cos_theta_c_n, evaluate=False)

def Y_s2_l2_m0():
    str_Y_s2_l2_m0 = '(3/4)*sqrt(5/(6*pi))*sin(theta_e)**2'
    return sympify(str_Y_s2_l2_m0, evaluate=False)

def RQF_SW():
    str_RQF_SW = f'(2* D_psi_dec - 3/2)*5*sqrt(6)/16*sin(theta_e)**2*\
    (A/r_H*(3/4*Delta_chi_dec*({Delta_cos_theta_c_n(4)})\
                     + (chi_e*cos(theta_e)-chi_c)*({Delta_cos_theta_c_n(3)}) \
                     -1/2*Delta_chi_dec*({Delta_cos_theta_c_n(2)}) - \
                     (chi_e*cos(theta_e) - chi_c)* ({Delta_cos_theta_c_n(1)})) + \
    B/(r_H**2)*(3/5*Delta_chi_dec**2*({Delta_cos_theta_c_n(5)}) + \
                          3/2*Delta_chi_dec*(chi_e*cos(theta_e)-chi_c)*({Delta_cos_theta_c_n(4)}) + \
                          1/3*(-Delta_chi_dec**2 + \
                               3*(chi_e*cos(theta_e)-chi_c)**2)*({Delta_cos_theta_c_n(3)}) - \
                          Delta_chi_dec*(chi_e*cos(theta_e)-chi_c)*({Delta_cos_theta_c_n(2)}) - \
                          (chi_e*cos(theta_e)-chi_c)**2*({Delta_cos_theta_c_n(1)})))'

    return sympify(str_RQF_SW, evaluate=False)

def RQF_Dopp():
    str_RQF_Dopp = f'D_v_dec* 5*sqrt(6)/16*sin(theta_e)**2*\
    (A/r_H*(3/4*({Delta_cos_theta_c_n(4)}) \
    - 1/2*({Delta_cos_theta_c_n(2)})) \
    + 2*B/(r_H**2)*(3/5*Delta_chi_dec*({Delta_cos_theta_c_n(5)}) + \
    3/4*(chi_e*cos_theta_e-chi_c)*({Delta_cos_theta_c_n(4)}) -\
    1/3*Delta_chi_dec*({Delta_cos_theta_c_n(3)}) - \
    1/2*(chi_e*cos_theta_e-chi_c)*({Delta_cos_theta_c_n(2)})))'

    return sympify(str_RQF_Dopp, evaluate=False)

def q_s2_l2_m0_SW_sym():
    return integrate(2*pi*sin(theta_e)*Y_s2_l2_m0()*RQF_SW(), theta_e)

def q_s2_l2_m0_Dopp_sym():
    return integrate(2*pi*sin(theta_e)*Y_s2_l2_m0()*RQF_Dopp(), theta_e)

def q_s2_l2_m0_SW(A_num, B_num, r_H_num, D_psi_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, theta_e_num):
    A = Symbol('A')
    B = Symbol('B')
    r_H = Symbol('r_H')
    chi_c = Symbol('chi_c')
    Delta = Symbol('Delta')
    chi_dec = Symbol('chi_dec')
    Delta_chi_dec = Symbol('Delta_chi_dec')

    chi_e = Symbol('chi_e')
    theta_e = Symbol('theta_e')
    D_psi_dec = Symbol('D_psi_dec')
    
    q_l2m0_lambda = lambdify((A, B, r_H, D_psi_dec, chi_c, chi_e, Delta_chi_dec, theta_e), \
                             q_s2_l2_m0_SW_sym())
    
    return q_l2m0_lambda(A_num, B_num, r_H_num, D_psi_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, np.pi) - \
    q_l2m0_lambda(A_num, B_num, r_H_num, D_psi_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, 0)

def q_s2_l2_m0_Dopp(A_num, B_num, r_H_num, D_v_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, theta_e_num):
    A = Symbol('A')
    B = Symbol('B')
    r_H = Symbol('r_H')
    chi_c = Symbol('chi_c')
    Delta = Symbol('Delta')
    chi_dec = Symbol('chi_dec')
    Delta_chi_dec = Symbol('Delta_chi_dec')

    chi_e = Symbol('chi_e')
    theta_e = Symbol('theta_e')
    D_v_dec = Symbol('D_v_dec')
    
    q_l2m0_lambda = lambdify((A, B, r_H, D_v_dec, chi_c, chi_e, Delta_chi_dec, theta_e), \
                             q_s2_l2_m0_Dopp_sym())
    
    return q_l2m0_lambda(A_num, B_num, r_H_num, D_v_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, np.pi) - \
    q_l2m0_lambda(A_num, B_num, r_H_num, D_v_dec_num, chi_c_num, chi_e_num, Delta_chi_dec_num, 0)
    
    
# Cos_theta_c()
# integrate(Cos_theta_c(), theta_e)
# Delta_cos_theta_c_n(1)
# RQF_SW()
# 2*pi*Y_s2_l2_m0()*RQF_SW()
# q_l2m0_sym = integrate(2*pi*sin(theta_e)*Y_s2_l2_m0()*RQF_SW(), theta_e)
# q_l2m0_sym

In [71]:
q_s2_l2_m0_SW_sym()

15*sqrt(5)*sqrt(pi)*(2*D_psi_dec - 3/2)*(-A*Delta_chi_dec*sin(theta_e)**4*cos(theta_e)/(4*r_H) - A*Delta_chi_dec*sin(theta_e)**2*cos(theta_e)**3/(3*r_H) - 2*A*Delta_chi_dec*cos(theta_e)**5/(15*r_H) + A*chi_c**2*sin(theta_e)**4*cos(theta_e)/(2*Delta_chi_dec*r_H) + 2*A*chi_c**2*sin(theta_e)**2*cos(theta_e)**3/(3*Delta_chi_dec*r_H) + 4*A*chi_c**2*cos(theta_e)**5/(15*Delta_chi_dec*r_H) + A*chi_c*chi_e*sin(theta_e)**6/(6*Delta_chi_dec*r_H) + A*chi_e**2*sin(theta_e)**4*cos(theta_e)**3/(6*Delta_chi_dec*r_H) + 2*A*chi_e**2*sin(theta_e)**2*cos(theta_e)**5/(15*Delta_chi_dec*r_H) + 4*A*chi_e**2*cos(theta_e)**7/(105*Delta_chi_dec*r_H) - A*chi_c**4*sin(theta_e)**4*cos(theta_e)/(4*Delta_chi_dec**3*r_H) - A*chi_c**4*sin(theta_e)**2*cos(theta_e)**3/(3*Delta_chi_dec**3*r_H) - 2*A*chi_c**4*cos(theta_e)**5/(15*Delta_chi_dec**3*r_H) - A*chi_c**3*chi_e*sin(theta_e)**6/(6*Delta_chi_dec**3*r_H) - A*chi_c**2*chi_e**2*sin(theta_e)**4*cos(theta_e)**3/(2*Delta_chi_dec**3*r_H) - 2*A*chi_c**2*chi_e**2*sin(theta_e)

In [72]:
q_s2_l2_m0_SW(A, B, r_H, D_psi_dec, x_c, r_e, r_edec, np.pi)
q_s2_l2_m0_Dopp(A, B, r_H, D_v_dec, x_c, r_e, r_edec, np.pi)

0.23890814082304104

0.0168893873698332